In [12]:
!pip install geopandas folium

In [13]:
import pandas as pd
import geopandas as gpd
import folium
import zipfile
import os

In [15]:
from google.colab import files

uploaded = files.upload()

Saving spatial_predictions.csv to spatial_predictions.csv


In [16]:
with zipfile.ZipFile(
    "hyderabad_orr_1km_mastergrid.zip",
    "r"
) as zip_ref:
    zip_ref.extractall("grid")
os.listdir("grid")

['hyderabad_orr_1km_mastergrid.shx',
 'hyderabad_orr_1km_mastergrid.prj',
 'hyderabad_orr_1km_mastergrid.cpg',
 'hyderabad_orr_1km_mastergrid.dbf',
 'hyderabad_orr_1km_mastergrid.shp']

In [17]:
pred = pd.read_csv(
    "spatial_predictions.csv"
)

pred.head()
pred.columns

Index(['grid_id', 'spatial_block', 'temperature_c', 'wind_speed_m_s',
       'population_density_persons_per_km2', 'road_density_km_per_km2',
       'no2_mol_m2', 'predicted_no2'],
      dtype='object')

In [18]:
grid = gpd.read_file(
    "grid/hyderabad_orr_1km_mastergrid.shp"
)

grid.head()

,id,left,top,right,bottom,row_index,col_index,grid_id,geometry
0,7,206391.0724,1.942644e+06,207391.0724,1.941644e+06,6,0,HYD_0001,"POLYGON ((78.23431 17.55096, 78.24372 17.55109..."
1,8,206391.0724,1.941644e+06,207391.0724,1.940644e+06,7,0,HYD_0002,"POLYGON ((78.23445 17.54193, 78.24386 17.54206..."
2,9,206391.0724,1.940644e+06,207391.0724,1.939644e+06,8,0,HYD_0003,"POLYGON ((78.23459 17.5329, 78.244 17.53303, 7..."
3,10,206391.0724,1.939644e+06,207391.0724,1.938644e+06,9,0,HYD_0004,"POLYGON ((78.23472 17.52387, 78.24413 17.524, ..."
4,11,206391.0724,1.938644e+06,207391.0724,1.937644e+06,10,0,HYD_0005,"POLYGON ((78.23486 17.51484, 78.24427 17.51497..."


In [19]:
pred["grid_id"] = (
    pred["grid_id"]
    .astype(str)
    .str.strip()
)

grid["grid_id"] = (
    grid["grid_id"]
    .astype(str)
    .str.strip()
)

In [20]:
common = set(pred.grid_id).intersection(
    set(grid.grid_id)
)

print(
    "Prediction cells:",
    len(pred.grid_id.unique())
)

print(
    "Grid cells:",
    len(grid.grid_id.unique())
)

print(
    "Matching:",
    len(common)
)

Prediction cells: 46
Grid cells: 1543
Matching: 46


In [21]:
gdf = grid.merge(
    pred,
    on="grid_id",
    how="inner"
)

print(len(gdf))

46


In [33]:
# Cell 10 — Version 1.1 Interactive NO₂ Map (Fixed)

import folium
import branca.colormap as cm


# -------------------------
# Prepare Data
# -------------------------

gdf_projected = gdf.to_crs(epsg=32644)

centroid = gdf_projected.geometry.centroid

centroid_wgs84 = gpd.GeoSeries(
    centroid,
    crs="EPSG:32644"
).to_crs(epsg=4326)

center = [
    centroid_wgs84.y.mean(),
    centroid_wgs84.x.mean()
]


# Ensure numeric values

for col in [
    "predicted_no2",
    "no2_mol_m2"
]:
    gdf[col] = pd.to_numeric(
        gdf[col],
        errors="coerce"
    )


# Error calculation

gdf["error"] = (
    gdf["no2_mol_m2"]
    -
    gdf["predicted_no2"]
)


# Scale for visualization

gdf["predicted_scaled"] = gdf["predicted_no2"] * 1e5
gdf["actual_scaled"] = gdf["no2_mol_m2"] * 1e5
gdf["error_scaled"] = gdf["error"] * 1e5



# -------------------------
# Create Map
# -------------------------

m = folium.Map(
    location=center,
    zoom_start=11
)



# -------------------------
# Color Functions
# -------------------------

from branca.colormap import linear


pred_colormap = linear.YlOrRd_09.scale(
    gdf["predicted_scaled"].min(),
    gdf["predicted_scaled"].max()
)

actual_colormap = linear.YlGnBu_09.scale(
    gdf["actual_scaled"].min(),
    gdf["actual_scaled"].max()
)

error_colormap = linear.RdBu_11.scale(
    gdf["error_scaled"].min(),
    gdf["error_scaled"].max()
)



# -------------------------
# Predicted Layer
# -------------------------

pred_layer = folium.FeatureGroup(
    name="Predicted NO₂",
    show=True
)


folium.GeoJson(
    gdf.to_json(),
    style_function=lambda feature: {
        "fillColor": pred_colormap(
            feature["properties"]["predicted_scaled"]
        ),
        "color": "black",
        "weight": 0.3,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "grid_id",
            "predicted_no2"
        ],
        aliases=[
            "Grid ID:",
            "Predicted NO₂:"
        ]
    )
).add_to(pred_layer)


pred_layer.add_to(m)



# -------------------------
# Actual Layer
# -------------------------

actual_layer = folium.FeatureGroup(
    name="Actual NO₂",
    show=False
)


folium.GeoJson(
    gdf.to_json(),
    style_function=lambda feature: {
        "fillColor": actual_colormap(
            feature["properties"]["actual_scaled"]
        ),
        "color": "black",
        "weight": 0.3,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "grid_id",
            "no2_mol_m2"
        ],
        aliases=[
            "Grid ID:",
            "Actual NO₂:"
        ]
    )
).add_to(actual_layer)


actual_layer.add_to(m)



# -------------------------
# Error Layer
# -------------------------

error_layer = folium.FeatureGroup(
    name="Prediction Error",
    show=False
)


folium.GeoJson(
    gdf.to_json(),
    style_function=lambda feature: {
        "fillColor": error_colormap(
            feature["properties"]["error_scaled"]
        ),
        "color": "black",
        "weight": 0.3,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "grid_id",
            "error"
        ],
        aliases=[
            "Grid ID:",
            "Error:"
        ]
    )
).add_to(error_layer)


error_layer.add_to(m)



# -------------------------
# Add Layer Control
# -------------------------

folium.LayerControl().add_to(m)


# Add legends

pred_colormap.caption = "Predicted NO₂ ×10⁻⁵"
pred_colormap.add_to(m)

actual_colormap.caption = "Actual NO₂ ×10⁻⁵"
actual_colormap.add_to(m)

error_colormap.caption = "Error ×10⁻⁵"
error_colormap.add_to(m)



m